# Installation Test Segment Anything

## Installation test

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from segment_anything import SamPredictor, sam_model_registry
import os

def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))

# 1. Initialize the model
sam_checkpoint = os.path.expanduser("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/SAM/ViT-H/sam_vit_h_4b8939.pth")
model_type = "vit_h"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
predictor = SamPredictor(sam)

# 2. Load the image
image_path = 'MAX_stk_0005_CA_SSU_GAPDH_Infection_1_CY5, CY3.5 NAR, FITC, DAPI, BF.jpg'
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 3. Set the image for the predictor
predictor.set_image(image)

# # 4. Define a bounding box as input
# input_box = np.array([100, 100, 600, 600]) # Example bounding box [x1, y1, x2, y2]

# Helper function to visualize the points
def show_points(coords, labels, ax, marker_size=100):
    """Draws point prompts on an image."""
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

# 4. Define a regular grid of points to cover the entire image
# Get image dimensions
height, width, _ = image.shape

# Define grid parameters
grid_points_per_side = 50

# Generate grid coordinates across the whole image
x_coords = np.linspace(0, width - 1, grid_points_per_side)
y_coords = np.linspace(0, height - 1, grid_points_per_side)
yv, xv = np.meshgrid(y_coords, x_coords)

# Format the points and labels for the predictor
input_points = np.stack([xv.ravel(), yv.ravel()], axis=-1)
input_labels = np.ones(input_points.shape[0], dtype=int)

# 5. Predict
masks, _, _ = predictor.predict(
    point_coords=input_points,
    box=None,
    point_labels=input_labels,
    multimask_output=True,
)

# # 5. Predict with bounding box
# masks, _, _ = predictor.predict(
#     point_coords=None,
#     point_labels=None,
#     box=input_box[None, :],
#     multimask_output=False,
# )

In [ ]:
# 6. Visualize the result
plt.figure(figsize=(10, 10))
plt.imshow(image)
show_mask(masks[0], plt.gca())
show_points(input_points, input_labels, plt.gca())
plt.axis('off')
plt.show()

In [ ]:
# Trying the automatic predictor

from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
mask_generator = SamAutomaticMaskGenerator(sam)
masks = mask_generator.generate(image)

In [ ]:
def show_anns(anns):
    """
    Displays the output of the SamAutomaticMaskGenerator.

    Args:
        anns (list): A list of masks, where each mask is a dictionary.
    """
    if len(anns) == 0:
        return
    
    # Sort annotations by area in descending order to draw larger masks first
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    
    ax = plt.gca()
    ax.set_autoscale_on(False)

    # Create a new image layer for the masks with transparency
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:,:,3] = 0 # Set alpha channel to 0 (fully transparent)

    for ann in sorted_anns:
        m = ann['segmentation']
        # Generate a random color for the mask
        color_mask = np.concatenate([np.random.random(3), [0.35]]) # RGBA with 0.35 alpha
        img[m] = color_mask
    
    ax.imshow(img)

plt.figure(figsize=(20,20))
plt.imshow(image)
show_anns(masks)
plt.title(f"SAM Automatic Masks - {len(masks)} objects found")
plt.axis('off')
plt.show()

## Roan data

Try SAM automatic segmentor on Roan's data FOV1

### Load files

In [ ]:
from pathlib import Path

# Define the path using pathlib
# folder_path = Path("~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/FOV1/Channels_Split").expanduser()
folder_path = Path("~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/FOV2/CA_SSU_GAPDH_Infection_1_CY5, CY3.5 NAR, FITC, DAPI_split").expanduser()

print(f"Checking path: {folder_path}")

# 1. Check if the folder actually exists
if not folder_path.is_dir():
    print("\n--- ERROR ---")
    print("The specified folder does not exist or is not a directory.")
    print("Please double-check the path.")
else:
    print("\nFolder exists. Listing contents...")
    
    # 2. List all files in the directory to see what's actually there
    files_in_dir = list(folder_path.iterdir())
    
    if not files_in_dir:
        print("The folder is empty.")
    else:
        print("--- Files found in directory: ---")
        for f in files_in_dir:
            print(f.name) # Print just the filename for clarity
        print("---------------------------------")

    # 3. Try the glob again and see if it finds anything
    # We convert it to a list to easily check if it's empty
    max_tif_files = sorted(folder_path.glob("SHARPEST*.tif"))
    # max_tif_files = sorted(folder_path.glob("MAX*.tif"))
    
    print(f"\nFound {len(max_tif_files)} files matching 'MAX*.tif'.")
    if not max_tif_files:
        print("Suggestion: Check for typos or case-sensitivity issues (e.g., are the files named 'max...' instead of 'MAX...')?")

### Run model and save results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from pathlib import Path
import torch
import os
from segment_anything import sam_model_registry, SamPredictor
from preprocessing import preprocess_image

def show_single_mask_on_ax(ax, mask, image):
    """Overlays a single boolean mask on a given Matplotlib axis with a random color."""
    overlay = np.ones((*image.shape[:2], 4))
    overlay[:, :, 3] = 0
    color = np.concatenate([np.random.random(3), [0.6]])
    overlay[mask] = color
    ax.imshow(image)
    ax.imshow(overlay)

def show_points_on_ax(ax, points, image):
    """Overlays point prompts on a given Matplotlib axis."""
    ax.imshow(image)
    ax.scatter(points[:, 0], points[:, 1], color='green', marker='*', s=150, edgecolor='white', linewidth=1.2)

def save_segmentation_visualization(
    processed_image: np.ndarray,
    prompts: np.ndarray,
    prompt_type: str,
    masks: list,
    scores: np.ndarray,
    logits: list,
    output_path: Path,
    title: str
):
    """
    Creates and saves a 3x3 grid showing the original image, prompt,
    the 3 generated masks, and their corresponding logits.
    """
    if not (len(masks) == len(scores) == len(logits) == 3):
        print(f"  Warning: Expected 3 masks, scores, and logits for visualization. Skipping.")
        return

    fig, axes = plt.subplots(3, 3, figsize=(24, 24))

    # --- Top Row: Image and Prompt ---
    axes[0, 0].imshow(processed_image)
    axes[0, 0].set_title("Original Processed Image", fontsize=14)
    
    if prompt_type == 'mask':
        axes[0, 1].imshow(prompts, cmap='gray')
        axes[0, 1].set_title("Input Prompt Mask", fontsize=14)
    elif prompt_type == 'points':
        show_points_on_ax(axes[0, 1], prompts, processed_image)
        axes[0, 1].set_title("Input DAPI Point Prompts", fontsize=14)
    
    # Hide the remaining empty plots in the top row
    axes[0, 2].axis('off')

    # --- Middle and Bottom Rows: Masks and Logits ---
    # Sort everything by score in descending order to show the best one first
    sorted_data = sorted(zip(masks, scores, logits), key=lambda x: x[1], reverse=True)
    
    for i, (mask, score, logit) in enumerate(sorted_data):
        # Middle row: Masks
        ax_mask = axes[1, i]
        show_single_mask_on_ax(ax_mask, mask, processed_image)
        ax_mask.set_title(f"Mask (Score: {score:.3f})", fontsize=14)

        # Bottom row: Logits
        ax_logit = axes[2, i]
        # Display the logit map. 'viridis' is a good colormap for this.
        im = ax_logit.imshow(logit, cmap='viridis')
        fig.colorbar(im, ax=ax_logit) # Add a colorbar to show the logit value scale
        ax_logit.set_title("Raw Logit Output", fontsize=14)

    # --- Final Touches and Saving ---
    for ax in axes.flatten():
        ax.axis('off')
    
    fig.suptitle(title, fontsize=22)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    
    plt.savefig(output_path)
    plt.close(fig)

# --- Setup ---
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/SAM/ViT-L/sam_vit_l_0b3195.pth").expanduser()
model_type = "vit_l"
# sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/SAM/ViT-H/sam_vit_h_4b8939.pth").expanduser()
# model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)

# --- Main Processing Loop ---
all_processed_images = {}
all_prompts = {}
all_prompt_types = {}
all_generated_masks = {}
all_generated_scores = {}
all_generated_logits = {}

# --- CHOOSE YOUR PROMPT MODE ---
# Option 1: Mask-based prompting from the main image
# PROMPT_MODE = 'mask' 
# DAPI_FILE_PATTERN = None # Not needed

# Option 2: Point-based prompting from DAPI images
PROMPT_MODE = 'points'
DAPI_FILE_INDEX = 3

dapi_path = None
if PROMPT_MODE == 'points' and DAPI_FILE_INDEX:
    # Fetches the corresponding DAPI file path
    dapi_path = max_tif_files[DAPI_FILE_INDEX]

print(f"Starting mask generation for {len(max_tif_files)} files using '{PROMPT_MODE}' prompts...")

for file_path in max_tif_files:
    file_name = file_path.name
    print(f"Processing {file_name}...")

    # 1. Preprocess the image and generate prompts
    processed_image, prompts, prompt_type = preprocess_image(
        file_path,
        scale_intensity=True,
        resize_image=True,
        max_dim=1024,
        convert_to_rgb=True,
        generate_prompt_mask=True,
        prompt_mode=PROMPT_MODE,
        dapi_image_path=dapi_path,
        min_mask_area=150,
        use_watershed=True
    )

    if processed_image is None or prompts is None:
        print(f"  Skipping {file_name} due to preprocessing/prompting error.")
        continue

    # Store results for visualization
    all_processed_images[file_name] = processed_image
    all_prompts[file_name] = prompts
    all_prompt_types[file_name] = prompt_type

    # 2. Set the image for the predictor
    predictor.set_image(processed_image)

    # 3. Predict masks based on the prompt type
    masks, scores, logits = (None, None, None) # Initialize
    
    if prompt_type == 'mask':
        prompt_mask_resized = cv2.resize(prompts, (256, 256), interpolation=cv2.INTER_NEAREST)
        mask_input = torch.as_tensor(prompt_mask_resized, dtype=torch.float32, device=predictor.device).unsqueeze(0).unsqueeze(0)
        masks, scores, logits = predictor.predict_torch(point_coords=None, point_labels=None, mask_input=mask_input, multimask_output=True)
    
    elif prompt_type == 'points':
        # Convert numpy arrays to torch tensors and add a batch dimension
        point_coords_tensor = torch.as_tensor(prompts, dtype=torch.float32, device=predictor.device).unsqueeze(0)
        point_labels_tensor = torch.as_tensor(np.ones(len(prompts)), dtype=torch.float32, device=predictor.device).unsqueeze(0)
        
        masks, scores, logits = predictor.predict_torch(
            point_coords=point_coords_tensor, 
            point_labels=point_labels_tensor, 
            multimask_output=True # Ensure we get 3 masks for visualization
        )
    
    # 4. Save the outputs for visualization
    if masks is not None:
        all_generated_masks[file_name] = [m.cpu().numpy() for m in masks[0]]
        all_generated_scores[file_name] = scores[0].cpu().numpy()
        all_generated_logits[file_name] = [l.cpu().numpy() for l in logits[0]]
        print(f"  Generated {len(all_generated_masks[file_name])} masks for {file_name}.")
    else:
        print(f"  Mask generation failed for {file_name}.")


print("\nFinished generating masks for all files.")

# --- Visualization Loop ---
output_dir = Path(f"output/SAM_{PROMPT_MODE}_prompts")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\nComparison visualizations will be saved to: {output_dir.resolve()}")

for file_path in max_tif_files:
    file_name = file_path.name

    if file_name not in all_generated_masks:
        continue

    print(f"Creating visualization for {file_name}...")
    
    output_filename = output_dir / f"{file_path.stem}_visualization.png"
    
    save_segmentation_visualization(
        processed_image=all_processed_images[file_name],
        prompts=all_prompts[file_name],
        prompt_type=all_prompt_types[file_name],
        masks=all_generated_masks[file_name],
        scores=all_generated_scores[file_name],
        logits=all_generated_logits[file_name],
        output_path=output_filename,
        title=f"SAM Output for {file_name}"
    )
    
    print(f"  -> Saved to {output_filename}")

print("\nAll comparison visualizations have been saved.")

In [ ]:
from load_files import find_files_by_pattern

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SUBTRACT*.tif"
    )

max_tif_files = find_files_by_pattern(search_path[2], file_pattern[1], verbose=True)

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import torch
import os
from segment_anything import sam_model_registry, SamPredictor
from preprocessing import preprocess_image
from visualize import save_channel_comparison_visualization

# --- Setup ---
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/SAM/ViT-L/sam_vit_l_0b3195.pth").expanduser()
model_type = "vit_l"
# sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/SAM/ViT-H/sam_vit_h_4b8939.pth").expanduser()
# model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)

# --- Main Processing Loop ---
all_processed_images = {}
all_prompts = {}
all_prompt_types = {}
all_generated_masks = {}
all_generated_scores = {}
all_generated_logits = {}

# --- CHOOSE YOUR PROMPT MODE ---
PROMPT_MODE = 'bbox' # Can be 'points', 'mask', or 'bbox'
DAPI_FILE_INDEX = 3

dapi_path = None
# Dynamically find the index for the DAPI file (e.g., C4-Composite.tif)
try:
    DAPI_FILE_INDEX = [i for i, f in enumerate(max_tif_files) if 'C5' in f.name][0]
except IndexError:
    raise FileNotFoundError("DAPI file with 'C5' in its name not found in the channel files.")

if PROMPT_MODE in ('points', 'bbox') and DAPI_FILE_INDEX is not None:
    # Fetches the corresponding DAPI file path
    dapi_path = max_tif_files[DAPI_FILE_INDEX]

print(f"Starting mask generation for {len(max_tif_files)} files using '{PROMPT_MODE}' prompts...")

for file_path in max_tif_files:
    file_name = file_path.name
    print(f"Processing {file_name}...")

    # 1. Preprocess the image and generate prompts
    processed_image, prompts, prompt_type = preprocess_image(
        file_path,
        scale_intensity=True,
        resize_image=True,
        max_dim=1024,
        convert_to_rgb=True,
        prompt_mode=PROMPT_MODE,
        dapi_image_path=dapi_path,
        min_mask_area=150,
        use_watershed=True,
        bbox_radius_multiplier=2
    )

    if processed_image is None or prompts is None:
        print(f"  Skipping {file_name} due to preprocessing/prompting error.")
        continue

    # Store results for visualization
    all_processed_images[file_name] = processed_image
    all_prompts[file_name] = prompts
    all_prompt_types[file_name] = prompt_type

    # 2. Set the image for the predictor
    predictor.set_image(processed_image)

    # 3. Predict masks based on the prompt type
    masks, scores, logits = (None, None, None) # Initialize
    
    # SAM's predictor is not designed for batching, so we loop through prompts.
    # This is different from microSAM's batched_inference.
    output_masks, output_scores, output_logits = [], [], []

    if prompt_type == 'points':
        for point in prompts:
            point_coords = torch.as_tensor([point], dtype=torch.float32, device=predictor.device).unsqueeze(0)
            point_labels = torch.as_tensor([1], dtype=torch.float32, device=predictor.device).unsqueeze(0)
            
            # Predict for a single point, getting the best mask (multimask_output=False)
            mask, score, logit = predictor.predict_torch(point_coords, point_labels, multimask_output=False)
            output_masks.append(mask[0, 0].cpu().numpy()) # Take the single mask and convert to numpy
            output_scores.append(score[0, 0].cpu().item())
            output_logits.append(logit[0, 0].cpu().numpy())

    elif prompt_type == 'bbox':
        for box in prompts:
            box_torch = torch.as_tensor(box, dtype=torch.float32, device=predictor.device).unsqueeze(0)
            
            mask, score, logit = predictor.predict_torch(point_coords=None, point_labels=None, boxes=box_torch, multimask_output=False)
            output_masks.append(mask[0, 0].cpu().numpy())
            output_scores.append(score[0, 0].cpu().item())
            output_logits.append(logit[0, 0].cpu().numpy())

    masks, scores, logits = output_masks, output_scores, output_logits
    
    # 4. Save the outputs for visualization
    if masks is not None:
        all_generated_masks[file_name] = masks
        all_generated_scores[file_name] = scores
        all_generated_logits[file_name] = logits
        print(f"  Generated {len(all_generated_masks[file_name])} masks for {file_name}.")
    else:
        print(f"  Mask generation failed for {file_name}.")


print("\nFinished generating masks for all files.")

# --- Visualization Loop ---
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path(f"output/SAM_{PROMPT_MODE}_prompts_comparison")
print(f"\nComparison visualizations will be saved to: {base_output_dir.resolve()}")

# Since we are creating one visualization per FOV (Field of View), we only need to run the loop once.
# We assume all files in `max_tif_files` belong to the same FOV.
if max_tif_files:
    # 1. Determine the relative path of the input file's folder.
    # We use the first file to determine the directory structure.
    first_file_path = max_tif_files[0]
    try:
        relative_dir = first_file_path.parent.relative_to(base_input_dir)
    except ValueError:
        relative_dir = first_file_path.parent.name
    
    # 2. Create the full output directory and the final filename for the combined plot.
    current_output_dir = base_output_dir / relative_dir
    current_output_dir.mkdir(parents=True, exist_ok=True)
    # The output filename can be based on the directory name (e.g., FOV1)
    output_filename = current_output_dir / f"{first_file_path.parent.name}_channel_comparison.png"
    
    print(f"Creating combined channel visualization for {relative_dir}...")
    
    # Call the new visualization function that handles all channels at once
    save_channel_comparison_visualization(
        all_processed_images=all_processed_images,
        all_prompts=all_prompts,
        all_prompt_types=all_prompt_types,
        all_generated_masks=all_generated_masks,
        all_generated_scores=all_generated_scores,
        all_generated_logits=all_generated_logits,
        file_paths=max_tif_files,
        output_path=output_filename,
        title=f"SAM Channel Comparison for {relative_dir}"
    )
    
    print(f"  -> Saved to {output_filename}")

print("\nAll comparison visualizations have been saved.")

## Appendix

### Compare with/without scaling intensities

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# 1. Load the high bit-depth image "as is"
# base_path = Path("~/Projects/C_Albicans Thesis Project/7. Data/Channels_Split/").expanduser()
base_path = Path("~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/FOV1/Channels_Split").expanduser()
img_path = base_path / "C3-Composite.tif"

print(f"Attempting to load image from: {img_path}")
# cv2.imread requires the path to be a string
image_16bit = cv2.imread(str(img_path))

if image_16bit is None:
    print("Error: Could not load the image.")
else:
    # 2. Find the actual minimum and maximum pixel values in the image
    min_val = np.min(image_16bit)
    max_val = np.max(image_16bit)

    print(f"Original bit depth: {image_16bit.dtype}")
    print(f"Actual pixel value range: [{min_val}, {max_val}]")

    # 3. Normalize the image to the 0-255 range (for 8-bit display)
    if max_val > min_val:
        image_8bit_scaled = cv2.normalize(image_16bit, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    else:
        image_8bit_scaled = np.zeros(image_16bit.shape, dtype=np.uint8)

    # 4. Display the images using Matplotlib (Jupyter-friendly)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Display original 16-bit image
    axes[0].imshow(image_16bit, cmap='gray')
    axes[0].set_title('Original 16-bit (Low Contrast Display)')
    axes[0].axis('off') # Hide axes ticks

    # Display scaled 8-bit image
    axes[1].imshow(image_8bit_scaled, cmap='gray')
    axes[1].set_title('Manually Scaled 8-bit (High Contrast)')
    axes[1].axis('off') # Hide axes ticks

    plt.tight_layout()
    plt.show()

### Check dimensions

The max projection files are 16 bit grayscale (width, height, 1 channel).

In [ ]:
import cv2

for f in max_tif_files:
    image = cv2.imread(str(f),cv2.IMREAD_UNCHANGED)
    print(image.ndim,image.shape)

### Clear VRAM

In [ ]:
import torch
import gc

def clear_gpu_memory(*vars_to_delete):
    """
    Safely deletes specified variables and then clears the GPU cache.
    
    Args:
        *vars_to_delete: A tuple of variable objects to be deleted.
                         It's best to pass the variables themselves.
    """
    # Note: This function is a bit of a placeholder for the concept.
    # The actual deletion must happen in the scope where the variables live.
    # The example below shows the correct pattern.
    
    # Run Python's garbage collector
    gc.collect()
    
    # Empty the CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Start of Cleanup ---

print("Attempting to clean up GPU resources...")

# Use a try/except block for each variable that might not exist
try:
    del predictor
    print("  - Deleted 'predictor'")
except NameError:
    pass # Variable didn't exist, do nothing.

try:
    del sam
    print("  - Deleted 'sam'")
except NameError:
    pass

try:
    del masks
    print("  - Deleted 'masks'")
except NameError:
    pass

try:
    del scores
    print("  - Deleted 'scores'")
except NameError:
    pass

try:
    del logits
    print("  - Deleted 'logits'")
except NameError:
    pass

# Finally, run garbage collection and empty the PyTorch cache
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared.")

print("Cleanup complete.")

# Display GPU memory usage at the end
free_mem, total_mem = torch.cuda.mem_get_info()
print(f"\nGPU Memory: {free_mem / 1024**2:.2f} MiB free / {total_mem / 1024**2:.2f} MiB total")

In [ ]:
from tifffile import TiffFile, imwrite

input_tif_path = Path("~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/FOV2/CA_SSU_GAPDH_Infection_1_CY5, CY3.5 NAR, FITC, DAPI.tif").expanduser()
with TiffFile(input_tif_path) as tif:
    # Read the entire series as a NumPy array
    data = tif.series[0].asarray()
    axes = tif.series[0].axes
    print(f"  Detected TIFF axes: {axes}, with shape: {data.shape}")